# Agent Memory — Demo (issue #75 / #37)

Each agent now carries a private **memory**: an append-only, timestamped log of
what it *observed* and *did*. This is the foundation the Generative Agents paper
(Park et al., 2023) builds perception, reflection, and planning on top of.

Memory is deliberately distinct from **knowledge** (#45, notebook `04`):

* **Memory** is the episodic *log* — "on turn 1 the player gave me a fish".
* **Knowledge** is the current *world-model* — "the tower door is locked".

Memory is **context, not authority**: a remembered line can steer which command
an agent picks, but the world graph stays the single source of truth and every
command still passes the parser's precondition gate. Memory is also **private** —
one agent's records never leak into another agent's prompt, into
`command_history`, or into `Game.events`.

This notebook is fully **offline and deterministic** (no API key). It reuses the
real Action Castle cast via `build_game()` and a scripted mock LLM, and shows:

1. A memory is a **timestamped record**; the stream is append-only.
2. **Retrieval** ranks memories by *recency × importance × relevance*.
3. **Perception** folds the *visible* slice of `Game.events` into memory.
4. Memory rides the **live ReAct loop**: a failed action becomes a memory, and a
   relevant memory flows back into the next turn's prompt.
5. **Privacy**: one agent's memories never reach another agent's prompt or history.
6. **Semantic relevance** (#76): swap keyword overlap for embeddings — opt-in, and
   off by default so everything above stays byte-identical.

In [1]:
from rich.console import Console

from hw1_solution.action_castle import build_game
from text_adventure_games.memory import AgentMemory

# A rich console for colored notebook output. force_jupyter=True makes rich emit
# its HTML here (a notebook's stdout isn't a TTY, so it would otherwise go plain).
console = Console(force_jupyter=True, width=100)


def banner(title):
    """A colored section rule, matching the look of the engine's turn headers."""
    console.rule(title, style="bold cyan", align="left")


def show_records(records):
    """Print a memory stream as its raw records (markup=False keeps the [..]
    labels literal). This is the stream itself; the prompt-format "Relevant
    memories:" block — what an agent actually reads — shows up in sections 2 & 4.
    """
    for r in records:
        console.print(
            f" • [{r.kind.value}, turn {r.created_turn}, importance {r.importance:.0f}] {r.text}",
            markup=False,
        )


# Build the canonical Action Castle. We won't run full rounds — we drive the
# memory stream and the ReAct loop by hand — so the game is just a convenient
# pre-populated world: a hungry troll guarding the drawbridge, a guard nearby.
game = build_game()
troll = game.characters["troll"]
console.print("Characters:", list(game.characters.keys()))
console.print("The troll waits on the", troll.location.name)

Characters:
['The player', 'troll', 'guard', 'princess', 'ghost']

The troll waits on the Drawbridge

## 1. A memory is a timestamped record

Each agent owns an `AgentMemory` — an **append-only** stream of `MemoryRecord`s.
A record carries its `kind` (observation / reflection / plan), the `text`, the
`turn` it was created on, and an `importance` on the paper's 1–10 "poignancy"
scale (1 = mundane, 10 = momentous). The writers (`add_observation`,
`add_reflection`, `add_plan`) only ever *append* — records are never edited or
deleted, so memory stays an honest log of what happened.

In [2]:
mem = AgentMemory(owner="troll")
mem.add_observation("The player crossed the drawbridge heading east.", turn=1)
mem.add_observation("The player gave me a tasty fish to eat.", turn=1, importance=8)
mem.add_observation("The guard shouted at me from the courtyard.", turn=2, importance=3)

# Appending assigns each record the next id, in order — the stream only grows.
console.print("record ids (always increasing):", [r.id for r in mem.records])

first = mem.records[0]
console.print(
    f"first record -> kind={first.kind.value!r}  turn={first.created_turn}  "
    f"importance={first.importance}"
)

banner("The whole stream, oldest first")
show_records(mem.records)

record ids (always increasing):
[0, 1, 2]

first record -> kind='observation'  turn=1  importance=1.0

The whole stream, oldest first ─────────────────────────────────────────────────────────────────────

• [observation, turn 1, importance 1] The player crossed the drawbridge heading east.

• [observation, turn 1, importance 8] The player gave me a tasty fish to eat.

• [observation, turn 2, importance 3] The guard shouted at me from the courtyard.

## 2. Retrieval — recency × importance × relevance

A run can pile up far more memories than fit in a prompt, so `retrieve(query,
turn)` scores every record and returns only the most useful few. The score is the
sum of three deterministic ingredients (equal-weighted, like the paper) — no model
and no network required:

| Ingredient | What it rewards | How it's computed |
|---|---|---|
| **recency** | memories touched recently | `0.95 ** (turns since last access)` |
| **importance** | momentous memories | the 1–10 score, normalized to 0–1 |
| **relevance** | memories about the query | keyword overlap with the query text |

Below, four memories compete. We print each ingredient *before* retrieving (so the
recency decay is visible — `retrieve` refreshes the records it returns), then let
`retrieve` surface the top three.

In [3]:
from text_adventure_games.memory import (
    importance_score,
    recency_score,
    relevance_score,
)

mem = AgentMemory(owner="troll")
mem.add_observation("The player gave me a tasty fish to eat.", turn=1, importance=8)
mem.add_observation("A bird flew over the drawbridge.", turn=1, importance=1)
mem.add_observation("The guard shouted at me from the courtyard.", turn=2, importance=3)
mem.add_observation("I am hungry and looking for food.", turn=5, importance=5)

query = "I am hungry. Is there any food or fish nearby?"
turn = 6

# Score every record now, before retrieve() refreshes anything.
scored = [
    (recency_score(r, turn), importance_score(r), relevance_score(query, r.text), r)
    for r in mem.records
]

banner(f"Per-ingredient scores for the query, at turn {turn}")
for rec, imp, rel, r in sorted(scored, key=lambda s: s[0] + s[1] + s[2], reverse=True):
    console.print(
        f"total={rec + imp + rel:.2f}   "
        f"recency={rec:.2f}  importance={imp:.2f}  relevance={rel:.2f}   {r.text!r}"
    )

console.print()
banner("retrieve(query, turn, max_records=3) keeps only the top three")
console.print(mem.render(mem.retrieve(query, turn=turn, max_records=3)), markup=False)

Per-ingredient scores for the query, at turn 6 ─────────────────────────────────────────────────────

total=1.88   recency=0.95  importance=0.50  relevance=0.43   'I am hungry and looking for food.'

total=1.72   recency=0.77  importance=0.80  relevance=0.14   'The player gave me a tasty fish to 
eat.'

total=1.11   recency=0.81  importance=0.30  relevance=0.00   'The guard shouted at me from the 
courtyard.'

total=0.87   recency=0.77  importance=0.10  relevance=0.00   'A bird flew over the drawbridge.'

retrieve(query, turn, max_records=3) keeps only the top three ──────────────────────────────────────

Relevant memories:
 - [observation, turn 5] I am hungry and looking for food.
 - [observation, turn 1] The player gave me a tasty fish to eat.
 - [observation, turn 2] The guard shouted at me from the courtyard.

## 3. Perception — folding visible events into memory

Agents don't only remember their own actions; they notice what happens around
them. `ingest_events(game, character)` walks the new entries in the shared
`Game.events` log and stores the ones this character could plausibly perceive:

* an event by someone in the **same room** (a co-located action), or
* an event whose payload **names** this character (it was about them).

Two things are deliberately left out: events in **other rooms**, and the agent's
**own** actions (those are captured more richly as outcomes by the ReAct loop —
see §4). Each kept event becomes one short sentence built from the event's
*command* text, never its third-person narration — so a remembered line can never
spoof another agent's decision.

In [4]:
game = build_game()
troll = game.characters["troll"]

# Put the player on the drawbridge, beside the troll; the guard stays away in
# the courtyard. Now we log three events and see which ones the troll perceives.
player = game.player
player.location.remove_character(player)
troll.location.add_character(player)

mem = AgentMemory(owner="troll")
game.log_event("The player", "go", summary="go east")        # co-located -> perceived
game.log_event("guard", "sharpen", summary="sharpen sword")  # other room  -> ignored
game.log_event("troll", "growl", summary="growl player")     # own action  -> skipped here

added = mem.ingest_events(game, troll)
console.print(f"{len(game.events)} events occurred; the troll perceived {len(added)} of them:")
show_records(mem.records)

3 events occurred; the troll perceived 1 of them:

• [observation, turn 0, importance 1] The player did: go east

## 4. Memory in the live ReAct loop

Now the payoff: memory woven into `react_behavior`, the Observe → Decide → Act →
Reflect loop. Each turn the agent first **perceives** new events, **retrieves** the
memories most relevant to its situation and folds them into the prompt, then
**acts** — and the *outcome* of that action (success or failure) is stored as a
fresh memory.

We drive the troll with a scripted mock LLM, so the story is deterministic and
free. On **turn 1** it tries `go south` — the drawbridge has no exit south, so the
action fails. On **turn 2** it tries `go west` — and we inspect its prompt to find
the remembered failure waiting there, ready to steer it.

In [5]:
from text_adventure_games.llm_client import MockLlmClient
from text_adventure_games.npc import LLMAgent, react_behavior
from text_adventure_games.webapp.web_parser import WebParser

game = build_game()
game.set_parser(WebParser(game))  # buffers narration instead of printing to stdout
troll = game.characters["troll"]

# A scripted brain: turn 1 it decides "go south" (fails), turn 2 "go west" (works).
mock = MockLlmClient(["go south", "go west"])
agent = LLMAgent(mock, persona="I am a hungry troll guarding the drawbridge.")

game.turn = 1
react_behavior(troll, game, agent, max_retries=0)
game.turn = 2
react_behavior(troll, game, agent, max_retries=0)

banner("The troll's memory after two turns (outcomes are captured automatically)")
show_records(agent.memory.records)

The troll's memory after two turns (outcomes are captured automatically) ───────────────────────────

• [observation, turn 1, importance 4] I tried "go south" but it failed because Drawbridge does not 
have an exit 'south'

• [observation, turn 2, importance 3] I tried "go west" and succeeded.

In [6]:
# What did the troll actually SEE on turn 2? The retrieved memory rides along in a
# "Relevant memories:" block, appended *after* the live observation (so it can
# never disturb the lines the parser and mock client scan).
turn2_prompt = mock.calls[-1]["messages"][-1]["content"]
memory_block = turn2_prompt[turn2_prompt.find("Relevant memories:") :]

banner("Turn-2 prompt — the remembered failure flows back in")
console.print(memory_block, markup=False)

Turn-2 prompt — the remembered failure flows back in ───────────────────────────────────────────────

Relevant memories:
 - [observation, turn 1] I tried "go south" but it failed because Drawbridge does not have an exit 
'south'

## 5. Privacy — one mind's memories stay its own

Memory is strictly per-agent. The troll above remembers a secret; the **guard**,
acting in the same game, has its own (empty) stream and never sees the troll's.
And because observation and outcome memories are written *only* to the agent's
private stream — never to `parser.command_history` or `Game.events` — they can't
leak into anyone else's observation either.

In [7]:
game = build_game()
game.set_parser(WebParser(game))
troll = game.characters["troll"]
guard = game.characters["guard"]

secret = "The player secretly gave me a golden fish."
troll_agent = LLMAgent(MockLlmClient(["look"]), persona="I am a troll.")
troll_agent.memory.owner = "troll"
troll_agent.memory.add_observation(secret, turn=0, importance=9)

# The guard takes a turn. It has no memories of its own and cannot see the troll's.
guard_mock = MockLlmClient(["look"])
guard_agent = LLMAgent(guard_mock, persona="I am a guard.")
react_behavior(guard, game, guard_agent)

guard_prompt = guard_mock.calls[-1]["messages"][-1]["content"]
history = " ".join(e["content"] for e in game.parser.command_history)

console.print("Is the troll's secret in the guard's prompt?", secret in guard_prompt)
console.print("Does the guard's prompt have a memory block?", "Relevant memories:" in guard_prompt)
console.print("Is the secret anywhere in command_history?", secret in history)

Is the troll's secret in the guard's prompt? False

Does the guard's prompt have a memory block? False

Is the secret anywhere in command_history? False

## 6. Semantic relevance — embeddings (issue #76)

Section 2's relevance was **keyword overlap**: a memory counts as relevant only if
it shares words with the query. So a paraphrase like *"the bridge guardian skipped
breakfast"* looks irrelevant to *"who is hungry?"* — there are no shared words.

Issue #76 makes relevance **pluggable**. Hand `AgentMemory` an *embedding client*
and `retrieve` scores relevance by **cosine similarity** between the query's and
each memory's embedding vector (rescaled to the same 0–1 range), instead of keyword
overlap. Recency, importance, privacy, and the empty-render guard are unchanged —
and it's **strictly opt-in**: with no client, relevance stays keyword overlap,
exactly as in section 2.

The backends sit behind one `EmbeddingClient` seam (`embedding_client.py`):

| Provider | `EMBEDDING_PROVIDER` | Install | Notes |
|---|---|---|---|
| **model2vec** (default) | `local` | `uv sync --extra embeddings` | free, offline, no torch |
| sentence-transformers | `sentence-transformers` | `uv sync --extra embeddings-st` | local transformer (torch) |
| OpenAI | `openai` | `uv sync --extra openai` + API key | hosted |
| mock | `mock` | built in | deterministic test stand-in |

The cell below uses the real **model2vec** model when it's installed — then
*"Who around here is hungry?"* retrieves the famished-guardian memory **with zero
shared words** (keyword relevance `0.00`). Without the extra it falls back to the
deterministic **mock** so this notebook still re-runs offline, asking with shared
vocabulary instead. The output says which backend ran.

In [8]:
from text_adventure_games.embedding_client import (
    LocalEmbeddingClient,
    MockEmbeddingClient,
)
from text_adventure_games.memory import relevance_score

# Use the real local model (model2vec) when it's installed; otherwise fall back to
# the deterministic mock so this cell still runs offline. The AgentMemory wiring is
# identical either way -- only the notion of "similar" changes.
try:
    embedder, real = LocalEmbeddingClient(), True  # downloads once, then offline
except Exception:
    embedder, real = MockEmbeddingClient(), False

mem = AgentMemory(owner="troll", embedding_client=embedder)
mem.add_observation(
    "The bridge guardian skipped breakfast and is famished.", turn=1, importance=5
)
mem.add_observation(
    "A merchant sold colorful silk at the market stall.", turn=1, importance=5
)
mem.add_observation("Rain fell softly on the castle roof at dusk.", turn=1, importance=5)

# A real model matches on *meaning*, so a query that shares no words still finds the
# famished guardian. The mock only matches literal words, so for it we ask with
# shared vocabulary -- the retrieval machinery is the same, the matching is simpler.
query = "Who around here is hungry?" if real else "Who skipped breakfast and is famished?"
turn = 1  # equal recency + importance, so relevance alone decides the ranking

banner(f"Backend: {'model2vec (real embeddings)' if real else 'mock stand-in (offline)'}")
console.print(f"Query: {query!r}\n")

console.print("Keyword relevance (the section 2 default) for each memory:")
for r in mem.records:
    console.print(f"  {relevance_score(query, r.text):.2f}   {r.text!r}", markup=False)

console.print("\nEmbedding retrieval surfaces the most relevant memory by meaning:")
console.print(mem.render(mem.retrieve(query, turn=turn, max_records=1)), markup=False)

if not real:
    console.print(
        "\n[dim]Install the embeddings extra (uv sync --extra embeddings) to run this "
        "with model2vec -- then even 'Who around here is hungry?', which shares no "
        "words with any memory, still finds the famished guardian.[/dim]"
    )

Backend: mock stand-in (offline) ───────────────────────────────────────────────────────────────────

Query: 'Who skipped breakfast and is famished?'

Keyword relevance (the section 2 default) for each memory:

0.75   'The bridge guardian skipped breakfast and is famished.'

0.00   'A merchant sold colorful silk at the market stall.'

0.00   'Rain fell softly on the castle roof at dusk.'

Embedding retrieval surfaces the most relevant memory by meaning:

Relevant memories:
 - [observation, turn 1] The bridge guardian skipped breakfast and is famished.

Install the embeddings extra (uv sync --extra embeddings) to run this with model2vec -- then even 
'Who around here is hungry?', which shares no words with any memory, still finds the famished 
guardian.

## How memory composes with the rest of the stack

Memory (#75) is the episodic layer beneath the Generative Agents loop; it sits
alongside **knowledge** (#45, notebook `04`) and feeds the prompt in its own
section:

| Layer | What it is | How it's filled | Observation section |
|---|---|---|---|
| **Knowledge** (#45) | the current world-model — possibly wrong | seeded up front or via `learn()` | "What you know:" |
| **Memory** (#75, this notebook) | the episodic log of what was seen/done | auto-captured each turn from outcomes + `Game.events` | "Relevant memories:" |

**Invariants.** Memory is *context, not authority*: the world graph stays the
single source of truth, and only actions through the precondition gate change the
world. An agent with nothing to recall produces a **byte-identical** observation to
before memory existed (the render is empty-guarded), so existing games are
unaffected. The stream also serializes via `to_primitive` / `from_primitive`, ready
to persist with its character.

**Still ahead** (design Stages 5–8 in `docs/design/agent-memory.md`): LLM-scored
importance, automatic **reflection** and **planning** that write back into the same
stream, and save/load wired through `Character`.